<a href="https://www.kaggle.com/code/avikdas567/predicting-student-test-scores?scriptVersionId=295170409" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv")
sub   = pd.read_csv("/kaggle/input/playground-series-s6e1/sample_submission.csv")

TARGET = "exam_score"
ID_COL = "id"

# -----------------------------
# Feature detection
# -----------------------------
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
features = [c for c in train.columns if c not in [TARGET, ID_COL]]

X = train[features]
y = train[TARGET]
X_test = test[features]

# -----------------------------
# Cross-validation setup
# -----------------------------
N_SPLITS = 3
SEED = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

test_preds = np.zeros(len(test))
TOTAL_FOLDS = N_SPLITS

# -----------------------------
# Training loop
# -----------------------------
for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    progress_pct = 100 * fold / TOTAL_FOLDS
    print(f"\n▶ Fold {fold}/{TOTAL_FOLDS}  (~{progress_pct:.0f}% done)")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=4,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=SEED,
        task_type="GPU",
        devices="0",
        early_stopping_rounds=100,
        verbose=200
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_cols
    )

    test_preds += model.predict(X_test) / TOTAL_FOLDS

# -----------------------------
# Save submission
# -----------------------------
sub["exam_score"] = test_preds
sub.to_csv("submission.csv", index=False)

print("\n submission.csv saved!")


▶ Fold 1/3  (~33% done)
0:	learn: 18.3169222	test: 18.2631474	best: 18.2631474 (0)	total: 164ms	remaining: 4m 5s
200:	learn: 8.8328451	test: 8.8255851	best: 8.8255851 (200)	total: 6.54s	remaining: 42.3s
400:	learn: 8.8085651	test: 8.8087951	best: 8.8087951 (400)	total: 12.9s	remaining: 35.3s
600:	learn: 8.7916746	test: 8.7994989	best: 8.7994989 (600)	total: 19.1s	remaining: 28.6s
800:	learn: 8.7781772	test: 8.7936213	best: 8.7936213 (800)	total: 25.4s	remaining: 22.2s
1000:	learn: 8.7660481	test: 8.7890340	best: 8.7890107 (996)	total: 31.7s	remaining: 15.8s
1200:	learn: 8.7550483	test: 8.7856967	best: 8.7856967 (1200)	total: 38s	remaining: 9.47s
1400:	learn: 8.7452401	test: 8.7831167	best: 8.7831167 (1400)	total: 44.4s	remaining: 3.14s
1499:	learn: 8.7402049	test: 8.7818281	best: 8.7818151 (1498)	total: 47.6s	remaining: 0us
bestTest = 8.781815085
bestIteration = 1498
Shrink model to first 1499 iterations.

▶ Fold 2/3  (~67% done)
0:	learn: 18.3029605	test: 18.2924320	best: 18.2924320 